In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Lab 12: LSTM Model for Time-Series Forecasting\n",
    "\n",
    "**Student Name:** M.Sufyan Khan  \n",
    "**Registration Number:** 22JZELE0477  \n",
    "**Course:** Machine Learning Lab  \n",
    "**Supervisor:** Engr. Irshad Ullah  \n",
    "**University:** UET Nowshera Campus  \n",
    "\n",
    "\n",
    "\n",
    "## Learning Objectives\n",
    "- Set the working directory and import Scikit-learn, TensorFlow/Keras, and custom time-series utilities.\n",
    "- Define an LSTM neural network architecture for sequential forecasting data.\n",
    "- Configure optimizer, model checkpoints, and training monitor callbacks.\n",
    "- Load training, validation, and testing CSV files for time-series prediction.\n",
    "- Evaluate LSTM forecasting output using MAE, MSE, RMSE, R2 score, and explained variance.\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Section 1: Working Directory and Library Imports\n",
    "This section sets the Lab 12 working directory and imports Scikit-learn metrics, TensorFlow/Keras layers, callbacks, optimizers, and custom time-series utility functions.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "os.chdir('D:\\Machine Learning\\ML Lab\\Lab 12')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score\n",
    "from timeseires.utils.to_split import to_split\n",
    "from timeseires.utils.multivariate_multi_step import multivariate_multi_step\n",
    "from timeseires.utils.multivariate_single_step import multivariate_single_step\n",
    "from timeseires.utils.univariate_multi_step import univariate_multi_step\n",
    "from timeseires.utils.univariate_single_step import univariate_single_step\n",
    "from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS\n",
    "from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint\n",
    "from tensorflow.keras.callbacks import ModelCheckpoint\n",
    "from timeseires.callbacks.TrainingMonitor import TrainingMonitor\n",
    "from tensorflow.keras.optimizers import Adam\n",
    "from tensorflow.keras.optimizers import SGD\n",
    "from tensorflow.keras.models import load_model\n",
    "from tensorflow.keras.layers import LSTM, Bidirectional, Add\n",
    "from tensorflow.keras.layers import BatchNormalization\n",
    "from tensorflow.keras.layers import Conv1D,TimeDistributed\n",
    "from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input\n",
    "from tensorflow.keras.models import Sequential,Model\n",
    "import pandas as pd\n",
    "import time, pickle\n",
    "import numpy as np\n",
    "import tensorflow.keras.backend as K\n",
    "import tensorflow\n",
    "from tensorflow.keras.layers import Input, Reshape, Lambda\n",
    "from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense\n",
    "from tensorflow.keras.regularizers import l2\n",
    "import glob\n",
    "import h5py\n",
    "import matplotlib.pyplot as plt\n",
    "from keras.callbacks import Callback"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 3,
   "metadata": {},
   "outputs": [],
   "source": [
    "#lookback = 24\n",
    "model = None\n",
    "start_epoch = 0\n",
    "time_steps=24\n",
    "num_features=21"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "metadata": {},
   "outputs": [],
   "source": [
    "def create_lstm():\n",
    "    input_data = Input(shape=(time_steps, num_features))\n",
    "    lstm_layer1 = LSTM(8, return_sequences=True)(input_data)\n",
    "    lstm_layer2 = LSTM(20)(lstm_layer1)\n",
    "    x = Flatten()(lstm_layer2)\n",
    "    output_data = Dense(1)(x)\n",
    "    model = Model(input_data, output_data)\n",
    "    return model"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Section 2: Model Parameters and LSTM Architecture\n",
    "The following cells define time steps, feature count, and the LSTM model architecture used for sequential forecasting.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 5,
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/html": [
       "<pre style=\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace\"><span style=\"font-weight: bold\">Model: \"functional\"</span>\n",
       "</pre>\n"
      ],
      "text/plain": [
       "\u001b[1mModel: \"functional\"\u001b[0m\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/html": [
       "<pre style=\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace\">┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓\n",
       "┃<span style=\"font-weight: bold\"> Layer (type)                    </span>┃<span style=\"font-weight: bold\"> Output Shape           </span>┃<span style=\"font-weight: bold\">       Param # </span>┃\n",
       "┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩\n",
       "│ input_layer (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">InputLayer</span>)        │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">24</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">21</span>)         │             <span style=\"color: #00af00; text-decoration-color: #00af00\">0</span> │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ lstm (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">LSTM</span>)                     │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">24</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">8</span>)          │           <span style=\"color: #00af00; text-decoration-color: #00af00\">960</span> │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ lstm_1 (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">LSTM</span>)                   │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">20</span>)             │         <span style=\"color: #00af00; text-decoration-color: #00af00\">2,320</span> │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ flatten (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">Flatten</span>)               │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">20</span>)             │             <span style=\"color: #00af00; text-decoration-color: #00af00\">0</span> │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ dense (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">Dense</span>)                   │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">1</span>)              │            <span style=\"color: #00af00; text-decoration-color: #00af00\">21</span> │\n",
       "└─────────────────────────────────┴────────────────────────┴───────────────┘\n",
       "</pre>\n"
      ],
      "text/plain": [
       "┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓\n",
       "┃\u001b[1m \u001b[0m\u001b[1mLayer (type)                   \u001b[0m\u001b[1m \u001b[0m┃\u001b[1m \u001b[0m\u001b[1mOutput Shape          \u001b[0m\u001b[1m \u001b[0m┃\u001b[1m \u001b[0m\u001b[1m      Param #\u001b[0m\u001b[1m \u001b[0m┃\n",
       "┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩\n",
       "│ input_layer (\u001b[38;5;33mInputLayer\u001b[0m)        │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m24\u001b[0m, \u001b[38;5;34m21\u001b[0m)         │             \u001b[38;5;34m0\u001b[0m │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ lstm (\u001b[38;5;33mLSTM\u001b[0m)                     │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m24\u001b[0m, \u001b[38;5;34m8\u001b[0m)          │           \u001b[38;5;34m960\u001b[0m │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ lstm_1 (\u001b[38;5;33mLSTM\u001b[0m)                   │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m20\u001b[0m)             │         \u001b[38;5;34m2,320\u001b[0m │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ flatten (\u001b[38;5;33mFlatten\u001b[0m)               │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m20\u001b[0m)             │             \u001b[38;5;34m0\u001b[0m │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ dense (\u001b[38;5;33mDense\u001b[0m)                   │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m1\u001b[0m)              │            \u001b[38;5;34m21\u001b[0m │\n",
       "└─────────────────────────────────┴────────────────────────┴───────────────┘\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/html": [
       "<pre style=\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace\"><span style=\"font-weight: bold\"> Total params: </span><span style=\"color: #00af00; text-decoration-color: #00af00\">3,301</span> (12.89 KB)\n",
       "</pre>\n"
      ],
      "text/plain": [
       "\u001b[1m Total params: \u001b[0m\u001b[38;5;34m3,301\u001b[0m (12.89 KB)\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/html": [
       "<pre style=\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace\"><span style=\"font-weight: bold\"> Trainable params: </span><span style=\"color: #00af00; text-decoration-color: #00af00\">3,301</span> (12.89 KB)\n",
       "</pre>\n"
      ],
      "text/plain": [
       "\u001b[1m Trainable params: \u001b[0m\u001b[38;5;34m3,301\u001b[0m (12.89 KB)\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/html": [
       "<pre style=\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace\"><span style=\"font-weight: bold\"> Non-trainable params: </span><span style=\"color: #00af00; text-decoration-color: #00af00\">0</span> (0.00 B)\n",
       "</pre>\n"
      ],
      "text/plain": [
       "\u001b[1m Non-trainable params: \u001b[0m\u001b[38;5;34m0\u001b[0m (0.00 B)\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "model1 = create_lstm()\n",
    "model1.summary()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 6,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "You must install pydot (`pip install pydot`) for `plot_model` to work.\n"
     ]
    }
   ],
   "source": [
    "tensorflow.keras.utils.plot_model(model1 )"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 7,
   "metadata": {},
   "outputs": [],
   "source": [
    "checkpoints = r'D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'\n",
    "OUTPUT_PATH = r'D:\\Machine Learning\\ML Lab\\Lab 12'\n",
    "FIG_PATH = os.path.sep.join([OUTPUT_PATH,\"\\history.png\"])\n",
    "JSON_PATH = os.path.sep.join([OUTPUT_PATH,\"\\history.json\"])"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 8,
   "metadata": {},
   "outputs": [],
   "source": [
    "# construct the callback to save only the *best* model to disk\n",
    "# based on the validation loss\n",
    "EpochCheckpoint1 = ModelCheckpoint(checkpoints,\n",
    "                             monitor=\"val_loss\",\n",
    "                             save_best_only=True, \n",
    "                             verbose=1)\n",
    "TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)\n",
    "\n",
    "# construct the set of callbacks\n",
    "callbacks = [EpochCheckpoint1,TrainingMonitor1]"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Section 3: Checkpoints, Callbacks, and Training Configuration\n",
    "This section prepares checkpoint paths, output directories, training monitor callbacks, optimizer settings, and model compilation.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 9,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "[INFO] compiling model...\n",
      "WARNING:tensorflow:TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.\n"
     ]
    }
   ],
   "source": [
    "# if there is no specific model checkpoint supplied, then initialize\n",
    "# the network and compile the model\n",
    "if model is None:\n",
    "    print(\"[INFO] compiling model...\")\n",
    "    model =create_lstm()\n",
    "    opt = Adam(1e-3)\n",
    "    model.compile(loss= 'mae', optimizer=opt, metrics=[\"mae\", \"mape\"])\n",
    "# otherwise, load the checkpoint from disk\n",
    "else:\n",
    "    print(\"[INFO] loading {}...\".format(model))\n",
    "    model = load_model(model)\n",
    "\n",
    "    # update the learning rate\n",
    "    print(\"[INFO] old learning rate: {}\".format(K.get_value(model.optimizer.lr)))\n",
    "    K.set_value(model.optimizer.lr, 1e-4)\n",
    "    print(\"[INFO] new learning rate: {}\".format(K.get_value(model.optimizer.lr)))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 10,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "D:\\Machine Learning\\ML Lab\\Lab 12\\AEP_train.csv => True\n",
      "D:\\Machine Learning\\ML Lab\\Lab 12\\AEP_validation.csv => True\n",
      "D:\\Machine Learning\\ML Lab\\Lab 12\\AEP_test.csv => True\n",
      "D:\\Machine Learning\\ML Lab\\Lab 12\\AEP_hourly.csv => True\n"
     ]
    },
    {
     "data": {
      "text/plain": [
       "((84907, 21), (24259, 21), (12130, 21), (121273, 2))"
      ]
     },
     "execution_count": 10,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "path_dataset = r'D:\\Machine Learning\\ML Lab\\Lab 12'\n",
    "\n",
    "path_tr = os.path.join(path_dataset, 'AEP_train.csv')\n",
    "path_v = os.path.join(path_dataset, 'AEP_validation.csv')\n",
    "path_te = os.path.join(path_dataset, 'AEP_test.csv')\n",
    "path_hourly = os.path.join(path_dataset, 'AEP_hourly.csv')\n",
    "\n",
    "for path in [path_tr, path_v, path_te, path_hourly]:\n",
    "    print(path, \"=>\", os.path.exists(path))\n",
    "\n",
    "df_tr = pd.read_csv(path_tr)\n",
    "train_set = df_tr.values\n",
    "\n",
    "df_v = pd.read_csv(path_v)\n",
    "validation_set = df_v.values\n",
    "\n",
    "df_te = pd.read_csv(path_te)\n",
    "test_set = df_te.values\n",
    "\n",
    "df_hourly = pd.read_csv(path_hourly)\n",
    "\n",
    "train_set.shape, validation_set.shape, test_set.shape, df_hourly.shape"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 11,
   "metadata": {},
   "outputs": [],
   "source": [
    "time_steps=24\n",
    "num_features=21"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 12,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Time Consumed 0.3776381015777588 sec\n"
     ]
    }
   ],
   "source": [
    "start = time.time()\n",
    "\n",
    "train_X, train_y = univariate_multi_step(\n",
    "    train_set,\n",
    "    time_steps,\n",
    "    target_col=0,\n",
    "    target_len=1\n",
    ")\n",
    "\n",
    "validation_X, validation_y = univariate_multi_step(\n",
    "    validation_set,\n",
    "    time_steps,\n",
    "    target_col=0,\n",
    "    target_len=1\n",
    ")\n",
    "\n",
    "test_X, test_y = univariate_multi_step(\n",
    "    test_set,\n",
    "    time_steps,\n",
    "    target_col=0,\n",
    "    target_len=1\n",
    ")\n",
    "\n",
    "print('Time Consumed', time.time() - start, \"sec\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Section 4: Dataset Loading and Forecast Preparation\n",
    "The following cells load train, validation, and test CSV files, then prepare the data for LSTM forecasting input/output format.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 13,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Epoch 1/10\n",
      "\u001b[1m2646/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m━\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0384 - mae: 0.0384 - mape: 677.2867\n",
      "Epoch 1: val_loss improved from None to 0.01867, saving model to D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-0001-loss0.02.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 1: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-0001-loss0.02.h5\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m20s\u001b[0m 7ms/step - loss: 0.0383 - mae: 0.0383 - mape: 675.6271 - val_loss: 0.0187 - val_mae: 0.0187 - val_mape: 9.4192\n",
      "Epoch 2/10\n",
      "\u001b[1m2644/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m━\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0164 - mae: 0.0164 - mape: 518.4185\n",
      "Epoch 2: val_loss improved from 0.01867 to 0.01505, saving model to D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-0002-loss0.02.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 2: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-0002-loss0.02.h5\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m16s\u001b[0m 6ms/step - loss: 0.0164 - mae: 0.0164 - mape: 516.7564 - val_loss: 0.0151 - val_mae: 0.0151 - val_mape: 7.7892\n",
      "Epoch 3/10\n",
      "\u001b[1m2644/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m━\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0120 - mae: 0.0120 - mape: 179.3001\n",
      "Epoch 3: val_loss improved from 0.01505 to 0.01131, saving model to D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-0003-loss0.01.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 3: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-0003-loss0.01.h5\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m16s\u001b[0m 6ms/step - loss: 0.0120 - mae: 0.0120 - mape: 178.7320 - val_loss: 0.0113 - val_mae: 0.0113 - val_mape: 5.6847\n",
      "Epoch 4/10\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0112 - mae: 0.0112 - mape: 260.4667\n",
      "Epoch 4: val_loss improved from 0.01131 to 0.01007, saving model to D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-0004-loss0.01.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 4: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-0004-loss0.01.h5\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m16s\u001b[0m 6ms/step - loss: 0.0112 - mae: 0.0112 - mape: 260.4667 - val_loss: 0.0101 - val_mae: 0.0101 - val_mape: 4.3204\n",
      "Epoch 5/10\n",
      "\u001b[1m2651/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m━\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0107 - mae: 0.0107 - mape: 187.5592\n",
      "Epoch 5: val_loss improved from 0.01007 to 0.00973, saving model to D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-0005-loss0.01.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 5: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-0005-loss0.01.h5\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m16s\u001b[0m 6ms/step - loss: 0.0107 - mae: 0.0107 - mape: 187.4504 - val_loss: 0.0097 - val_mae: 0.0097 - val_mape: 4.0989\n",
      "Epoch 6/10\n",
      "\u001b[1m2652/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m━\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0102 - mae: 0.0102 - mape: 167.0220\n",
      "Epoch 6: val_loss improved from 0.00973 to 0.00931, saving model to D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-0006-loss0.01.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 6: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-0006-loss0.01.h5\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m17s\u001b[0m 6ms/step - loss: 0.0102 - mae: 0.0102 - mape: 166.9874 - val_loss: 0.0093 - val_mae: 0.0093 - val_mape: 4.3436\n",
      "Epoch 7/10\n",
      "\u001b[1m2645/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m━\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0099 - mae: 0.0099 - mape: 106.7619\n",
      "Epoch 7: val_loss did not improve from 0.00931\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m18s\u001b[0m 7ms/step - loss: 0.0099 - mae: 0.0099 - mape: 106.4658 - val_loss: 0.0102 - val_mae: 0.0102 - val_mape: 4.4667\n",
      "Epoch 8/10\n",
      "\u001b[1m2645/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m━\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0093 - mae: 0.0093 - mape: 330.4374\n",
      "Epoch 8: val_loss did not improve from 0.00931\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m17s\u001b[0m 6ms/step - loss: 0.0093 - mae: 0.0093 - mape: 329.5027 - val_loss: 0.0096 - val_mae: 0.0096 - val_mape: 4.2849\n",
      "Epoch 9/10\n",
      "\u001b[1m2646/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m━\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0089 - mae: 0.0089 - mape: 399.4630\n",
      "Epoch 9: val_loss improved from 0.00931 to 0.00874, saving model to D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-0009-loss0.01.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 9: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-0009-loss0.01.h5\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m17s\u001b[0m 6ms/step - loss: 0.0089 - mae: 0.0089 - mape: 398.4812 - val_loss: 0.0087 - val_mae: 0.0087 - val_mape: 3.9902\n",
      "Epoch 10/10\n",
      "\u001b[1m2646/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m━\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0085 - mae: 0.0085 - mape: 187.4521\n",
      "Epoch 10: val_loss improved from 0.00874 to 0.00772, saving model to D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-0010-loss0.01.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 10: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-0010-loss0.01.h5\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m17s\u001b[0m 6ms/step - loss: 0.0085 - mae: 0.0085 - mape: 186.9946 - val_loss: 0.0077 - val_mae: 0.0077 - val_mape: 3.2515\n"
     ]
    }
   ],
   "source": [
    "epochs = 10\n",
    "verbose = 1\n",
    "batch_size = 32\n",
    "\n",
    "History = model.fit(\n",
    "    train_X,\n",
    "    train_y,\n",
    "    batch_size=batch_size,\n",
    "    epochs=epochs,\n",
    "    validation_data=(validation_X, validation_y),\n",
    "    callbacks=callbacks,\n",
    "    verbose=verbose\n",
    ")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 14,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "True\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "c:\\Users\\Yousaf Tahir\\anaconda3\\Lib\\site-packages\\sklearn\\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:\n",
      "https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations\n",
      "  warnings.warn(\n"
     ]
    }
   ],
   "source": [
    "import os\n",
    "import pickle\n",
    "\n",
    "path_dataset = r'D:\\Machine Learning\\ML Lab\\Lab 12'\n",
    "path_scaler = os.path.join(path_dataset, 'AEP_Scaler.pkl')\n",
    "\n",
    "print(os.path.exists(path_scaler))\n",
    "\n",
    "with open(path_scaler, 'rb') as f:\n",
    "    scaler = pickle.load(f)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 15,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\u001b[1m379/379\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m1s\u001b[0m 2ms/step\n",
      "y_pred_scaled.shape = (12105, 1)\n",
      "test_y.shape = (12105, 1)\n",
      "Mean Absolute Error (MAE): 125.2\n",
      "Median Absolute Error (MedAE): 100.05\n",
      "Mean Squared Error (MSE): 26510.53\n",
      "Root Mean Squared Error (RMSE): 162.82\n",
      "Mean Absolute Percentage Error (MAPE): 0.86 %\n",
      "Median Absolute Percentage Error (MDAPE): 0.69 %\n",
      "\n",
      "\n",
      "y_test_unscaled.shape =  (12105, 1)\n",
      "y_pred.shape =  (12105, 1)\n"
     ]
    }
   ],
   "source": [
    "model_path = r'D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-0010-loss0.01.h5'\n",
    "\n",
    "model = load_model(model_path, compile=False)\n",
    "\n",
    "y_pred_scaled = model.predict(test_X)\n",
    "\n",
    "print(\"y_pred_scaled.shape =\", y_pred_scaled.shape)\n",
    "print(\"test_y.shape =\", test_y.shape)\n",
    "\n",
    "def inverse_target(scaled_values, scaler, target_col=0, num_features=21):\n",
    "    scaled_values = scaled_values.reshape(-1)\n",
    "\n",
    "    dummy = np.zeros((scaled_values.shape[0], num_features))\n",
    "    dummy[:, target_col] = scaled_values\n",
    "\n",
    "    unscaled = scaler.inverse_transform(dummy)\n",
    "\n",
    "    return unscaled[:, target_col].reshape(-1, 1)\n",
    "\n",
    "y_pred = inverse_target(\n",
    "    y_pred_scaled,\n",
    "    scaler,\n",
    "    target_col=0,\n",
    "    num_features=21\n",
    ")\n",
    "\n",
    "y_test_unscaled = inverse_target(\n",
    "    test_y,\n",
    "    scaler,\n",
    "    target_col=0,\n",
    "    num_features=21\n",
    ")\n",
    "\n",
    "# Mean Absolute Error (MAE)\n",
    "MAE = np.mean(np.abs(y_pred - y_test_unscaled))\n",
    "print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))\n",
    "\n",
    "# Median Absolute Error (MedAE)\n",
    "MEDAE = np.median(np.abs(y_pred - y_test_unscaled))\n",
    "print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))\n",
    "\n",
    "# Mean Squared Error (MSE)\n",
    "MSE = np.mean(np.square(y_pred - y_test_unscaled))\n",
    "print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))\n",
    "\n",
    "# Root Mean Squared Error (RMSE)\n",
    "RMSE = np.sqrt(MSE)\n",
    "print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))\n",
    "\n",
    "# Mean Absolute Percentage Error (MAPE)\n",
    "MAPE = np.mean(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100\n",
    "print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')\n",
    "\n",
    "# Median Absolute Percentage Error (MDAPE)\n",
    "MDAPE = np.median(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100\n",
    "print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')\n",
    "\n",
    "print('\\n\\ny_test_unscaled.shape = ', y_test_unscaled.shape)\n",
    "print('y_pred.shape = ', y_pred.shape)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 16,
   "metadata": {},
   "outputs": [],
   "source": [
    "checkpoints = r'D:\\Machine Learning\\ML Lab\\Lab 12\\\\E2-cp-{epoch:04d}-loss{val_loss:.2f}.h5'\n",
    "model=r'D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-0010-loss0.01.h5'\n",
    "start_epoch= 34"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Section 5: Prediction and Model Evaluation\n",
    "The final cells generate predictions and calculate regression metrics to evaluate the forecasting model performance.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 17,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "[INFO] loading D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-0010-loss0.01.h5...\n",
      "[INFO] old learning rate: 9.999999747378752e-05\n",
      "[INFO] new learning rate: 9.999999747378752e-05\n"
     ]
    }
   ],
   "source": [
    "# construct the callback to save only the *best* model to disk\n",
    "# based on the validation loss\n",
    "EpochCheckpoint1 = ModelCheckpoint(checkpoints,\n",
    "                             monitor=\"val_loss\",\n",
    "                             save_best_only=True, \n",
    "                             verbose=1)\n",
    "TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)\n",
    "\n",
    "# construct the set of callbacks\n",
    "callbacks = [EpochCheckpoint1,TrainingMonitor1]\n",
    "model_load = r'D:\\Machine Learning\\ML Lab\\Lab 12\\\\E1-cp-0010-loss0.01.h5'\n",
    "# if there is no specific model checkpoint supplied, then initialize\n",
    "# the network and compile the model\n",
    "if model is None:\n",
    "    print(\"[INFO] compiling model...\")\n",
    "    model = PC.build(time_steps=24, num_features=21, reg=0.0005)\n",
    "    opt = Adam(1e-3)\n",
    "    model.compile(loss= 'mae', optimizer=opt, metrics=[\"mae\", \"mape\"])\n",
    "# otherwise, load the checkpoint from disk\n",
    "else:\n",
    "    print(\"[INFO] loading {}...\".format(model_load))\n",
    "    model = load_model(model_load, compile=False)\n",
    "\n",
    "    opt = Adam(learning_rate=1e-4)\n",
    "    model.compile(\n",
    "        loss=\"mae\",\n",
    "        optimizer=opt,\n",
    "        metrics=[\"mae\", \"mape\"]\n",
    "    )\n",
    "\n",
    "\n",
    "    # update the learning rate\n",
    "    print(\"[INFO] old learning rate: {}\".format(K.get_value(model.optimizer.learning_rate)))\n",
    "    print(\"[INFO] new learning rate: {}\".format(K.get_value(model.optimizer.learning_rate)))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 19,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Epoch 1/10\n",
      "\u001b[1m2650/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m━\u001b[0m \u001b[1m0s\u001b[0m 5ms/step - loss: 0.0073 - mae: 0.0073 - mape: 166.7622\n",
      "Epoch 1: val_loss improved from None to 0.00712, saving model to D:\\Machine Learning\\ML Lab\\Lab 12\\\\E2-cp-0001-loss0.01.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 1: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 12\\\\E2-cp-0001-loss0.01.h5\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m18s\u001b[0m 6ms/step - loss: 0.0073 - mae: 0.0073 - mape: 166.6031 - val_loss: 0.0071 - val_mae: 0.0071 - val_mape: 3.0111\n",
      "Epoch 2/10\n",
      "\u001b[1m2651/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m━\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0072 - mae: 0.0072 - mape: 210.6734\n",
      "Epoch 2: val_loss did not improve from 0.00712\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m17s\u001b[0m 6ms/step - loss: 0.0072 - mae: 0.0072 - mape: 210.5506 - val_loss: 0.0072 - val_mae: 0.0072 - val_mape: 3.0468\n",
      "Epoch 3/10\n",
      "\u001b[1m2652/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m━\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0072 - mae: 0.0072 - mape: 186.1138\n",
      "Epoch 3: val_loss did not improve from 0.00712\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m17s\u001b[0m 6ms/step - loss: 0.0072 - mae: 0.0072 - mape: 186.0748 - val_loss: 0.0072 - val_mae: 0.0072 - val_mape: 3.1171\n",
      "Epoch 4/10\n",
      "\u001b[1m2652/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m━\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0071 - mae: 0.0071 - mape: 164.5441\n",
      "Epoch 4: val_loss did not improve from 0.00712\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m17s\u001b[0m 6ms/step - loss: 0.0071 - mae: 0.0071 - mape: 164.5098 - val_loss: 0.0071 - val_mae: 0.0071 - val_mape: 3.0671\n",
      "Epoch 5/10\n",
      "\u001b[1m2648/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m━\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0071 - mae: 0.0071 - mape: 158.6682\n",
      "Epoch 5: val_loss did not improve from 0.00712\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m17s\u001b[0m 7ms/step - loss: 0.0071 - mae: 0.0071 - mape: 158.3988 - val_loss: 0.0073 - val_mae: 0.0073 - val_mape: 3.1888\n",
      "Epoch 6/10\n",
      "\u001b[1m2650/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m━\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0071 - mae: 0.0071 - mape: 161.6774\n",
      "Epoch 6: val_loss did not improve from 0.00712\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m17s\u001b[0m 7ms/step - loss: 0.0071 - mae: 0.0071 - mape: 161.5242 - val_loss: 0.0072 - val_mae: 0.0072 - val_mape: 3.0562\n",
      "Epoch 7/10\n",
      "\u001b[1m2646/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m━\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0070 - mae: 0.0070 - mape: 110.3182\n",
      "Epoch 7: val_loss did not improve from 0.00712\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m17s\u001b[0m 6ms/step - loss: 0.0070 - mae: 0.0070 - mape: 110.0506 - val_loss: 0.0074 - val_mae: 0.0074 - val_mape: 3.0981\n",
      "Epoch 8/10\n",
      "\u001b[1m2650/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m━\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0070 - mae: 0.0070 - mape: 134.7690\n",
      "Epoch 8: val_loss did not improve from 0.00712\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m17s\u001b[0m 6ms/step - loss: 0.0070 - mae: 0.0070 - mape: 134.6410 - val_loss: 0.0072 - val_mae: 0.0072 - val_mape: 3.1773\n",
      "Epoch 9/10\n",
      "\u001b[1m2649/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m━\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0070 - mae: 0.0070 - mape: 154.1849\n",
      "Epoch 9: val_loss did not improve from 0.00712\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m17s\u001b[0m 6ms/step - loss: 0.0070 - mae: 0.0070 - mape: 153.9800 - val_loss: 0.0071 - val_mae: 0.0071 - val_mape: 2.9718\n",
      "Epoch 10/10\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0070 - mae: 0.0070 - mape: 89.1392\n",
      "Epoch 10: val_loss improved from 0.00712 to 0.00702, saving model to D:\\Machine Learning\\ML Lab\\Lab 12\\\\E2-cp-0010-loss0.01.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 10: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 12\\\\E2-cp-0010-loss0.01.h5\n",
      "\u001b[1m2653/2653\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m17s\u001b[0m 6ms/step - loss: 0.0070 - mae: 0.0070 - mape: 89.1392 - val_loss: 0.0070 - val_mae: 0.0070 - val_mape: 3.0428\n"
     ]
    }
   ],
   "source": [
    "epochs = 10\n",
    "verbose = 1 #0\n",
    "batch_size = 32\n",
    "History = model.fit(train_X,\n",
    "                        train_y,\n",
    "                        batch_size=batch_size,   \n",
    "                        epochs = epochs, \n",
    "                        validation_data = (validation_X,validation_y),\n",
    "                        callbacks=callbacks,\n",
    "                        verbose = verbose)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 20,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\u001b[1m379/379\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m1s\u001b[0m 2ms/step\n",
      "Mean Absolute Error (MAE): 114.28\n",
      "Median Absolute Error (MedAE): 89.78\n",
      "Mean Squared Error (MSE): 22997.58\n",
      "Root Mean Squared Error (RMSE): 151.65\n",
      "Mean Absolute Percentage Error (MAPE): 0.78 %\n",
      "Median Absolute Percentage Error (MDAPE): 0.62 %\n",
      "\n",
      "\n",
      "y_test_unscaled.shape=  (12105, 1)\n",
      "y_pred.shape=  (12105, 1)\n"
     ]
    }
   ],
   "source": [
    "model = load_model(r'D:\\Machine Learning\\ML Lab\\Lab 12\\\\E2-cp-0010-loss0.01.h5', compile=False)\n",
    "\n",
    "y_pred_scaled   = model.predict(test_X)\n",
    "y_pred          = scaler.inverse_transform(y_pred_scaled)\n",
    "y_test_unscaled = scaler.inverse_transform(test_y)\n",
    "# Mean Absolute Error (MAE)\n",
    "MAE = np.mean(abs(y_pred - y_test_unscaled)) \n",
    "print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))\n",
    "\n",
    "# Median Absolute Error (MedAE)\n",
    "MEDAE = np.median(abs(y_pred - y_test_unscaled))\n",
    "print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))\n",
    "\n",
    "# Mean Squared Error (MSE)\n",
    "MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()\n",
    "print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))\n",
    "\n",
    "# Root Mean Squarred Error (RMSE) \n",
    "RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))\n",
    "print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))\n",
    "\n",
    "# Mean Absolute Percentage Error (MAPE)\n",
    "MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100\n",
    "print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')\n",
    "\n",
    "# Median Absolute Percentage Error (MDAPE)\n",
    "MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100\n",
    "print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')\n",
    "\n",
    "print('\\n\\ny_test_unscaled.shape= ',y_test_unscaled.shape)\n",
    "print('y_pred.shape= ',y_pred.shape)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Final Conclusion\n",
    "In this lab, an LSTM neural network was implemented for time-series forecasting. The notebook demonstrates sequence-model architecture, callback-based training, data preparation, and regression-based model evaluation.\n"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "base",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.13.9"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}